# Custom Climate Profiles Generation

#### Step 0: Set-Up
Import the [climakitae](https://github.com/cal-adapt/climakitae) library and other dependencies.

In [1]:
import pandas as pd
import xarray as xr
import numpy as np

from climakitae.explore.standard_year_profile import (
    get_climate_profile,
    export_profile_to_csv,
)
from climakitae.explore.typical_meteorological_year import TMY
from climakitae.core.data_interface import get_subsetting_options

from climakitae.explore.extreme_meteorological_year import (
    persistence_XMY,
    shock_XMY,
    persistence_get_top_hours)

from climakitae.core.constants import UNSET

from climakitae.explore.typical_meteorological_year import (
    get_cdf,
    get_cdf_monthly,
)


import warnings

warnings.filterwarnings("ignore")

### Generate an Extreme Meteorological Year (XMY) Profile

In [ ]:
start_year = 2005
end_year = 2020

In [ ]:
# Set up the TMY profile generator! The verbose option will output progress of the TMY generation
xmy = persistence_XMY(
    ## Location -- uncomment your desired option
    station_name = "Los Angeles International Airport (KLAX)",
    # latitude = latitude,
    # longitude = longitude,
    q = 0.9,
    
    # Approach -- uncomment your desired option
    warming_level = 1.5,
    # start_year = start_year,
    # end_year = end_year,
    verbose=True,
)

# Generate the profile!
xmy.generate_xmy()

**Export to non-EPW format.** TMY profiles are exported in .epw format by default, but can be exported as both `.csv` and `.tmy` file formats using the method `export_tmy_data` with the argument `extension="csv"` as shown below.

In [ ]:
xmy.export_xmy_data(extension="csv")

### Unit Testing

In [4]:
air_temp = xr.load_dataset("air_temp_lax_wl1_5_june.nc")

In [5]:
all_vars = xr.load_dataset("all_vars_lax_wl1_5_june.nc")

In [7]:
test_data = np.arange(0, 365 * 3 * 24, 1)
test_data = np.expand_dims(test_data, [1, 2])
coords = {
    "x": 7.819e05,
    "y": -4.116e06,
    "lakemask": 0,
    "landmask": 0,
    "Lambert_Conformal": 0,
    "time": pd.date_range(start="2001-01-01-00", end="2003-12-31-23", freq="1h"),
    "scenario": ["Historical + SSP 3-7.0"],
    "simulation": ["WRF_EC-Earth3_r1i1p1f1"],
}
da = xr.DataArray(
    name="Air Temperature at 2m",
    dims=["time", "scenario", "simulation"],
    data=test_data,
    coords=coords,
)
da.attrs = {
    "variable_id": "t2",
    "extended_description": "Temperature of the air 2m above Earth's surface.",
    "units": "degC",
    "data_type": "Gridded",
    "resolution": "9 km",
    "frequency": "hourly",
    "location_subset": ["coordinate selection"],
    "approach": "Time",
    "downscaling_method": "Dynamical",
    "institution": "UCLA",
    "grid_mapping": "Lambert_Conformal",
    "timezone": "America/Los_Angeles",
}

In [2]:
def mock_t_ds() -> xr.Dataset:
    """Fake hourly dataset that can be manipulated to set up tests."""
    test_data = np.arange(0, 365 * 3 * 24, 1)
    test_data = np.expand_dims(test_data, [1, 2])
    coords = {
        "x": 7.819e05,
        "y": -4.116e06,
        "lakemask": 0,
        "landmask": 0,
        "Lambert_Conformal": 0,
        "time": pd.date_range(start="2001-01-01-00", end="2003-12-31-23", freq="1h"),
        "scenario": ["Historical + SSP 3-7.0"],
        "simulation": ["WRF_EC-Earth3_r1i1p1f1"],
    }
    da = xr.DataArray(
        name="Air Temperature at 2m",
        dims=["time", "scenario", "simulation"],
        data=test_data,
        coords=coords,
    )
    da.attrs = {
        "variable_id": "t2",
        "extended_description": "Temperature of the air 2m above Earth's surface.",
        "units": "degC",
        "data_type": "Gridded",
        "resolution": "9 km",
        "frequency": "hourly",
        "location_subset": ["coordinate selection"],
        "approach": "Time",
        "downscaling_method": "Dynamical",
        "institution": "UCLA",
        "grid_mapping": "Lambert_Conformal",
        "timezone": "America/Los_Angeles",
    }
    return da.to_dataset()

In [13]:
data = {
    "month": list(range(1, 13)),
    "simulation": ["WRF_EC-Earth3_r1i1p1f1" for x in range(0, 12)],
    "year": [2001 for x in range(0, 12)],
}
df = pd.DataFrame.from_dict(data)
all_vars_ds = mock_t_ds()

stn_name = "Santa Ana John Wayne Airport (KSNA)"
start_year = 2001
end_year = 2003
q = 0.9
xmy = persistence_XMY(
    q=q,
    start_year=start_year,
    end_year=end_year,
    station_name=stn_name,
)
result = xmy._make_8760_tables(all_vars_ds, df)
# Check result dict of dataframes (only 1 for 1 simulation in test)
assert list(result.keys()) == list(all_vars_ds.simulation.values)
assert (
    result["WRF_EC-Earth3_r1i1p1f1"].columns
    == [
        "time",
        "scenario",
        "simulation",
        "x",
        "y",
        "lakemask",
        "landmask",
        "Lambert_Conformal",
        "Air Temperature at 2m",
    ]
).all()
assert len(result["WRF_EC-Earth3_r1i1p1f1"].index) == 8760

Initializing persistence XMY object for Santa Ana John Wayne Airport (KSNA).
Generating p90 persistence XMY.
Calculating persistence XMY for simulation: WRF_EC-Earth3_r1i1p1f1


KeyError: 'sim'